# Local implementation workflow and testing


In [2]:
# GloVe smoke test: pair scoring, paper loss, gradients, co-occurrences, and training.
from pathlib import Path
import sys

import torch

working_directory = Path.cwd()
project_candidates = [
    working_directory,
    *working_directory.parents,
    working_directory / "deep-learning-papers-pytorch",
]
project_root = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "implementations/models/glove.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the deep-learning-papers-pytorch project root")
sys.path.insert(0, str(project_root))

from implementations.models.glove import GloveModel, build_cooccurrence_matrix

torch.manual_seed(0)
vocab_size = 5
embedding_dim = 8
model = GloveModel(vocab_size=vocab_size, embedding_dim=embedding_dim)

# The model predicts dot(w_i, w_tilde_j) + b_i + b_tilde_j for observed pairs.
target_ids = torch.tensor([1, 1, 2, 3])
context_ids = torch.tensor([2, 3, 3, 1])
counts = torch.tensor([1.0, 0.5, 1.0, 0.5])
predictions = model(target_ids, context_ids)
assert predictions.shape == counts.shape

# Check the weighted squared-log-count objective and that gradients propagate.
loss = model.loss(target_ids, context_ids, counts)
assert loss.ndim == 0 and torch.isfinite(loss)
loss.backward()
assert all(
    parameter.grad is not None and torch.isfinite(parameter.grad).all()
    for parameter in model.parameters()
)

# A three-token sentence gives inverse-distance weights of 1 and 1/2.
cooccurrence = build_cooccurrence_matrix([[1, 2, 3]], vocab_size, window_size=2)
expected = torch.tensor(
    [
        [0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.5, 0.0],
        [0.0, 1.0, 0.0, 1.0, 0.0],
        [0.0, 0.5, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0],
    ]
)
assert torch.allclose(cooccurrence.to_dense(), expected)

# Run two AdaGrad epochs and verify finite objective history and vector shape.
history = model.fit(cooccurrence, epochs=2, shuffle=False)
embeddings = model.get_embeddings()
assert len(history) == 2 and torch.isfinite(torch.tensor(history)).all()
assert embeddings.shape == (vocab_size, embedding_dim)

print(f"Forward output shape: {tuple(predictions.shape)}")
print(f"Initial batch loss: {loss.item():.4f}")
print(f"Training objectives: {[round(value, 4) for value in history]}")
print(f"Combined embedding shape: {tuple(embeddings.shape)}")

c:\Users\Dell\Desktop\practice\deep-learning-papers-pytorch\implementations\models\glove.py:76: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:855.)
  return torch.sparse_coo_tensor(


Forward output shape: (4,)
Initial batch loss: 0.0179
Training objectives: [0.0182, 0.0135]
Combined embedding shape: (5, 8)


# colab implementation workflow


In [6]:
from pathlib import Path
import sys

import torch

working_directory = Path.cwd()
content_directory = Path("/content")
project_candidates = [working_directory, *working_directory.parents]
if content_directory.is_dir():
    project_candidates.extend(
        [content_directory]
        + [path for path in content_directory.iterdir() if path.is_dir()]
    )

project_root = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "implementations/models/glove.py").is_file()
    ),
    None,
)
if project_root is None:
    content_entries = (
        sorted(path.name for path in content_directory.iterdir())
        if content_directory.is_dir()
        else ["/content does not exist"]
    )
    raise RuntimeError(
        "Could not find implementations/models/glove.py. "
        f"Notebook cwd: {working_directory}; /content entries: {content_entries}. "
        "In VS Code Explorer, right-click the repository folder and select Upload to Colab."
    )

sys.path.insert(0, str(project_root))
print("Project root:", project_root)

if not torch.cuda.is_available():
    raise RuntimeError("Reconnect using a Colab server with a GPU")
print("GPU:", torch.cuda.get_device_name(0))

Project root: /content/deep-learning-papers-pytorch
GPU: Tesla T4


In [7]:
%pip install -q datasets

In [ ]:
# WikiText-2 raw: https://huggingface.co/datasets/Salesforce/wikitext
# Review the dataset card for its current license and citation requirements.
from datasets import load_dataset
from pathlib import Path
from implementations.models.glove import GloveModel
from implementations.utils.glove_data import load_glove_data

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
sample_lines = wiki["train"][:2000]["text"]
corpus_path = Path("/content/wikitext2_train_sample.txt")
corpus_path.write_text("\n".join(sample_lines), encoding="utf-8")

# The co-occurrence builder is CPU/Python-based; start with this bounded sample.
train_loader, vocabulary = load_glove_data(
    corpus_path,
    window_size=5,
    batch_size=1024,
    min_count=2,
    max_vocab_size=10000,
    shuffle=False,
)
cooccurrence = train_loader.dataset.cooccurrence_matrix
print(f"Vocabulary: {len(vocabulary):,}; observed pairs: {cooccurrence._nnz():,}")

# fit() uses AdaGrad internally and moves pair IDs/counts to the model device.
device = torch.device("cuda")
model = GloveModel(len(vocabulary), embedding_dim=100).to(device)
history = model.fit(
    cooccurrence,
    epochs=10,
    learning_rate=0.05,
    batch_size=1024,
)
print("Summed epoch objectives:", [round(value, 3) for value in history])

# Save to the uploaded repository's results folder on the Colab server.
artifact = {
    "state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    },
    "vocabulary": vocabulary,
    "embeddings": model.get_embeddings().detach().cpu(),
}
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, checkpoint_path)
print("Saved:", checkpoint_path)

Vocabulary: 6,709; observed pairs: 420,422
Summed epoch objectives: [15451.9847869873, 6353.9105968475, 4768.471145153, 3717.8605418205, 3001.7714185715, 2512.5643043518, 2160.9556438923, 1898.722751379, 1693.9343409538, 1529.4730615616]
Saved /content/glove_wikitext2_sample.pt


In [ ]:
# Continue the current model weights for 25 more epochs.
# fit() creates a new AdaGrad optimizer, so optimizer accumulators restart here.
continuation_history = model.fit(
    cooccurrence,
    epochs=25,
    learning_rate=0.05,
    batch_size=1024,
)
history.extend(continuation_history)
print("Objectives for continuation:", [round(value, 3) for value in continuation_history])
print("All epoch objectives:", [round(value, 3) for value in history])

# Refresh the repository results checkpoint with the continued model parameters.
artifact = {
    "state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    },
    "vocabulary": vocabulary,
    "embeddings": model.get_embeddings().detach().cpu(),
}
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, checkpoint_path)
print("Updated:", checkpoint_path)

Objectives for continuation: [1172.289, 1351.372, 472.732, 279.938, 208.55, 172.588, 150.894, 135.762, 124.604, 115.963, 108.959, 103.071, 98.006, 93.596, 89.63, 86.097, 82.878, 79.961, 77.257, 74.729, 72.426, 70.214, 68.164, 66.246, 64.421]
All epoch objectives: [15451.985, 6353.911, 4768.471, 3717.861, 3001.771, 2512.564, 2160.956, 1898.723, 1693.934, 1529.473, 2731.653, 1776.208, 1062.46, 824.755, 700.355, 615.287, 552.291, 502.733, 461.508, 427.051, 1504.098, 1450.745, 617.565, 419.332, 338.586, 292.931, 262.371, 239.661, 221.96, 207.262, 194.725, 183.978, 174.433, 165.916, 158.294, 1172.289, 1351.372, 472.732, 279.938, 208.55, 172.588, 150.894, 135.762, 124.604, 115.963, 108.959, 103.071, 98.006, 93.596, 89.63, 86.097, 82.878, 79.961, 77.257, 74.729, 72.426, 70.214, 68.164, 66.246, 64.421]
Updated /content/glove_wikitext2_sample.pt


In [16]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
drive_results_dir = Path("/content/drive/MyDrive/deep-learning-papers-pytorch/results")
drive_results_dir.mkdir(parents=True, exist_ok=True)
drive_checkpoint_path = drive_results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, drive_checkpoint_path)
print("Saved Drive backup:", drive_checkpoint_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved Drive backup: /content/drive/MyDrive/deep-learning-papers-pytorch/results/glove_wikitext2_sample.pt


In [14]:
from google.colab import files

# Save the current weights in the uploaded project before downloading.
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, checkpoint_path)
print("Colab checkpoint:", checkpoint_path)

# In the browser save dialog, choose the local deep-learning-papers-pytorch/results folder.
files.download(str(checkpoint_path))

Colab checkpoint: /content/deep-learning-papers-pytorch/results/glove_wikitext2_sample.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

To transfer it to the local project, download the file from the remote results folder:


6,709 vocabulary entries, 100-dimensional embeddings, and model weights matching the saved vectors. No need to download it again.

Next, inspect nearest neighbors to sanity-check the embeddings:


In [3]:
import torch
import torch.nn.functional as F

checkpoint = torch.load(
    "../results/glove_wikitext2_sample.pt",
    map_location="cpu",
    weights_only=False,
)
vocabulary = checkpoint["vocabulary"]
vectors = F.normalize(checkpoint["embeddings"], dim=1)
id_to_word = {index: word for word, index in vocabulary.items()}

def nearest_neighbors(word, count=10):
    if word not in vocabulary:
        print(f"{word!r} is not in the vocabulary")
        return
    word_id = vocabulary[word]
    scores = vectors @ vectors[word_id]
    scores[word_id] = -torch.inf
    for index in torch.topk(scores, min(count, len(vocabulary) - 1)).indices.tolist():
        print(id_to_word[index], round(scores[index].item(), 3))

nearest_neighbors("language")

wasp 0.523
pound 0.411
hodgkin 0.389
concert 0.385
carbenoid 0.376
eight 0.375
monte 0.371
themed 0.364
agrarian 0.357
anti 0.357


# resuls

Those are cosine similarities, not accuracy scores. The model ran, but neighbors like wasp, pound, and carbenoid are not convincing semantic neighbors for language, so these embeddings are noisy, not evidence of a strong result.

The likely limitation is the small, fixed sample: only 2,000 WikiText-2 lines and about 420,000 observed co-occurrence pairs. More epochs can lower the training objective, but they can’t supply missing context evidence. Try a larger, representative corpus sample and compare on held-out text or a word-similarity benchmark before judging quality.


# kaggle training
